In [2]:
# data loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#  data cleaning
import re
import string

# text processing 
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
# Ttransformers for NLP tasks
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import pipeline

#  models and training
import torch

# EVALUATION METRICS 
import evaluate
# rouge = evaluate.load("rouge")

# visualization
import plotly.express as px
from tqdm import tqdm              # Progress bar
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# data loading
df = pd.read_csv('data/merged_data.csv')  
df.head()

,title,word_count,content
0,The Open Medical-LLM Leaderboard: Benchmarking...,1359,"Datasets, Tasks, and Evaluation SetupMedQAMedM..."
1,Director of Machine Learning Insights [Part 2:...,2894,Omar Rahman1. How has ML made a positive impac...
2,Hands‑On with Agents SDK: Your First API‑Calli...,2387,"Publish AI, ML & data-science insights to a gl..."
3,4M Models Scanned: Protect AI + Hugging Face 6...,1250,Maintaining a Zero Trust Approach to Model Sec...
4,Judge Arena: Benchmarking LLMs as Evaluators,517,Judge ArenaHow it worksSelected ModelsThe Lead...


In [4]:
# data exploration function
def data_exploration(df):
    print("Data Overview:")
    print(df.info())
    print("\nFirst 5 Rows:")
    print(df.head())
    
    print("\nMissing Values:")
    print(df.isnull().sum())
    
    print("\nDescriptive Statistics:")
    print(df.describe())
data_exploration(df)

Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4452 entries, 0 to 4451
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       4452 non-null   object
 1   word_count  4452 non-null   int64 
 2   content     4452 non-null   object
dtypes: int64(1), object(2)
memory usage: 104.5+ KB
None

First 5 Rows:
                                               title  word_count  \
0  The Open Medical-LLM Leaderboard: Benchmarking...        1359   
1  Director of Machine Learning Insights [Part 2:...        2894   
2  Hands‑On with Agents SDK: Your First API‑Calli...        2387   
3  4M Models Scanned: Protect AI + Hugging Face 6...        1250   
4       Judge Arena: Benchmarking LLMs as Evaluators         517   

                                             content  
0  Datasets, Tasks, and Evaluation SetupMedQAMedM...  
1  Omar Rahman1. How has ML made a positive impac...  
2  Publish AI, ML & data-science 

In [5]:
def clean_text(text):
    """Clean text by removing special characters and extra spaces."""
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'[“”]', '"').replace("’", "'")   # Replace smart quotes with standard quotes
    return text.strip()
def preprocess_text(df, content_column):
    """Preprocess text data in the specified column."""
    df[content_column] = df[content_column].apply(lambda x: clean_text(x) if isinstance(x, str) else x)
    return df

In [6]:
df.head()

,title,word_count,content
0,The Open Medical-LLM Leaderboard: Benchmarking...,1359,"Datasets, Tasks, and Evaluation SetupMedQAMedM..."
1,Director of Machine Learning Insights [Part 2:...,2894,Omar Rahman1. How has ML made a positive impac...
2,Hands‑On with Agents SDK: Your First API‑Calli...,2387,"Publish AI, ML & data-science insights to a gl..."
3,4M Models Scanned: Protect AI + Hugging Face 6...,1250,Maintaining a Zero Trust Approach to Model Sec...
4,Judge Arena: Benchmarking LLMs as Evaluators,517,Judge ArenaHow it worksSelected ModelsThe Lead...


In [7]:
new_word_counts = df['content'].apply(lambda x: len(x.split()))
df['word_counts'] = new_word_counts

In [8]:
df.head()

,title,word_count,content,word_counts
0,The Open Medical-LLM Leaderboard: Benchmarking...,1359,"Datasets, Tasks, and Evaluation SetupMedQAMedM...",1000
1,Director of Machine Learning Insights [Part 2:...,2894,Omar Rahman1. How has ML made a positive impac...,1000
2,Hands‑On with Agents SDK: Your First API‑Calli...,2387,"Publish AI, ML & data-science insights to a gl...",1000
3,4M Models Scanned: Protect AI + Hugging Face 6...,1250,Maintaining a Zero Trust Approach to Model Sec...,1000
4,Judge Arena: Benchmarking LLMs as Evaluators,517,Judge ArenaHow it worksSelected ModelsThe Lead...,517


In [9]:
df.describe()

,word_count,word_counts
count,4452.000000,4452.000000
mean,604.442722,474.950584
std,777.170601,397.450482
min,100.000000,100.000000
25%,174.000000,174.000000
50%,252.000000,252.000000
75%,816.000000,816.000000
max,10519.000000,6051.000000


In [10]:
fig = px.histogram(df, x='word_counts', title='Word Count Distribution', labels={'word_counts': 'Number of Words'})
fig.update_layout(xaxis_title='Number of Words', yaxis_title='Frequency')
fig.show()

In [11]:
# Calculate IQR
Q1 = df['word_counts'].quantile(0.25)
Q3 = df['word_counts'].quantile(0.75)
IQR = Q3 - Q1

# Define outlier condition
outlier_condition = (df['word_counts'] < (Q1 - 1.5 * IQR)) | (df['word_counts'] > (Q3 + 1.5 * IQR))

# Count outliers
outliers = df[outlier_condition]
print(f"Outliers found: {len(outliers)}")

# Drop outliers
df_cleaned = df[~outlier_condition].reset_index(drop=True)

Outliers found: 31


In [12]:
vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words='english')
X = vectorizer.fit_transform(df['content'])  # Replace 'text_column' with your actual text column
sum_words = X.sum(axis=0)

# Create a DataFrame with bigrams and their frequencies
ngrams_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
ngrams_freq = sorted(ngrams_freq, key=lambda x: x[1], reverse=True)

# Display top 10 bigrams
top_ngrams = pd.DataFrame(ngrams_freq[:20], columns=["Bigram", "Frequency"])
print(top_ngrams)

                     Bigram  Frequency
0          machine learning       1629
1              hugging face       1589
2           language models        964
3               open source        802
4                 state art        772
5                real world        758
6            large language        660
7               fine tuning        615
8             deep learning        556
9           neural networks        477
10             data science        476
11              time series        463
12         natural language        434
13  artificial intelligence        433
14           neural network        420
15                real time        406
16                blog post        398
17            training data        383
18              pre trained        367
19               gives plot        365


In [13]:
df_cleaned.shape

(4421, 4)

In [14]:
fig = px.histogram(df_cleaned, x='word_counts', title='Word Count Distribution', labels={'word_counts': 'Number of Words'})
fig.update_layout(xaxis_title='Number of Words', yaxis_title='Frequency')
fig.show()

In [15]:
df_cleaned.columns

Index(['title', 'word_count', 'content', 'word_counts'], dtype='object')

In [16]:
# df = df_cleaned[['content']]  # keep relevant columns

# # Split into train and test (e.g., 80/20)
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# # Reset index for cleanliness
# train_df = train_df.reset_index(drop=True)
# test_df = test_df.reset_index(drop=True)

# # Check the shape and content
# print(train_df.shape)
# print(test_df[['content']].head())

In [17]:
# Load pre-trained BART model and tokenizer
# model_name = "facebook/bart-large-cnn"
# tokenizer = BartTokenizer.from_pretrained(model_name)
# model = BartForConditionalGeneration.from_pretrained(model_name)

# # Move model to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# # function to generate summary
# def generate_summary(text, max_input_length=1024, max_output_length=150):
#     # Tokenize the input text
#     inputs = tokenizer.encode(text, return_tensors="pt", max_length=max_input_length, truncation=True).to(device)
    
#     # Generate summary ids
#     summary_ids = model.generate(
#         inputs,
#         max_length=max_output_length,
#         num_beams=4,
#         length_penalty=2.0,
#         early_stopping=True
#     )
    
#     # Decode and return summary
#     return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# # apply to your test data
# tqdm.pandas()  # progress bar
# df_cleaned['generated_summary'] = df_cleaned['content'].progress_apply(generate_summary)



In [18]:
df_cleaned['summary_len'] = df_cleaned['generated_summary'].apply(lambda x: len(x.split()))
df_cleaned.head()

KeyError: 'generated_summary'

In [21]:
with_summaries_df = pd.read_csv('Summarized.csv')
with_summaries_df.tail()

,title,word_count,content,word_counts,generated_summary,summary_len
4416,Are Autonomous Agents the Future of AI-Powered...,815,The Data Scientist Artificial Intelligence has...,815,The Data Scientist Artificial Intelligence has...,54
4417,Guidance for NeurIPS Workshop Proposals 2025,302,Communications Chairs 20252025 Conference Auth...,302,NeurIPS 2025 workshops will be one-day in-pers...,52
4418,NeurIPS Announces Support for Newly Developing...,197,Communications Chairs 20252025 Conference We a...,197,"NeurIPS is endorsing EurIPS, a meeting taking ...",48
4419,Mistral AI gives Le Chat voice recognition and...,626,AI News is part of the TechForge Publications ...,626,Mistral AI has updated Le Chat with voice reco...,51
4420,Import AI,2945,"Welcome to Import AI, a newsletter about AI re...",1000,Stanford researchers have used test-time compu...,56


In [22]:
with_summaries_df.describe()

,word_count,word_counts,summary_len
count,4421.000000,4421.000000,4421.000000
mean,590.161728,459.761592,52.539697
std,756.961617,346.398078,9.897212
min,100.000000,100.000000,25.000000
25%,174.000000,174.000000,46.000000
50%,250.000000,250.000000,51.000000
75%,800.000000,800.000000,57.000000
max,10519.000000,1771.000000,127.000000
